# 06. Evaluate And Visualize

이 노트북은 Track B 결과를 공식 metric과 judge metric으로 정리하고 그림을 생성합니다.

- 각 prediction CSV를 gold case와 merge합니다.
- Accuracy, ERR, TP/FP/FN 등 official metric을 계산합니다.
- judge acceptability, confidence, social-context split을 요약합니다.
- 결과 요약 CSV와 figure를 생성합니다.

주요 출력:

- `outputs/context_rag/summary_pure.json`
- `outputs/context_rag/summary_pair.json`
- `outputs/context_rag/summary_metadata.json`
- `outputs/context_rag/combined_summary.csv`
- `outputs/context_rag/figures/*.png`


## 평가/시각화 설정

프로젝트 루트를 고정하고 결과 정리에 필요한 모듈을 import합니다. 이 노트북은 `evaluate_system_outputs()`와 `plot_from_input_dir()`를 직접 호출하며, 같은 작업은 `scripts/06_evaluate_outputs.py`, `scripts/06_visualize.py`로 command 실행할 수 있습니다.


In [ ]:
import pathlib, runpy

BOOTSTRAP = pathlib.Path("scripts/01_02_03_04_05_06_notebook_bootstrap.py")
if not BOOTSTRAP.exists():
    BOOTSTRAP = pathlib.Path("/content/lexnorm_submit/scripts/01_02_03_04_05_06_notebook_bootstrap.py")

setup_project = runpy.run_path(str(BOOTSTRAP))["setup_project"]
PROJECT_ROOT = setup_project()

from lexnorm.utils import sync_to_drive

import json
import pandas as pd

from lexnorm.evaluation import evaluate_system_outputs
from lexnorm.visualization import plot_from_input_dir

CONTEXT_DIR = pathlib.Path("outputs/context_rag")
EVAL_TASKS = [
    ("pure", CONTEXT_DIR / "preds_pure.csv", CONTEXT_DIR / "judge_pure.csv"),
    ("pair", CONTEXT_DIR / "preds_pair.csv", CONTEXT_DIR / "judge_pair.csv"),
    ("metadata", CONTEXT_DIR / "preds_metadata.csv", CONTEXT_DIR / "judge_metadata.csv"),
]


## official metric과 judge metric 요약

세 시스템의 prediction/judge CSV를 평가해서 summary JSON과 combined summary CSV를 만듭니다.


In [ ]:
summary_paths = []
for mode, pred_csv, judge_csv in EVAL_TASKS:
    out = CONTEXT_DIR / f"summary_{mode}.json"
    result = evaluate_system_outputs(
        cases_csv=CONTEXT_DIR / "audit_cases.csv",
        preds_csv=pred_csv,
        judge_csv=judge_csv,
        output_json=out,
    )
    print("saved", out)
    sync_to_drive(out)
    summary_paths.append(out)

rows = []
for path in summary_paths:
    obj = json.load(open(path, encoding="utf-8"))
    row = {"summary_path": str(path)}
    row.update(obj.get("official", {}))
    for key, value in obj.get("judge", {}).items():
        row[f"judge_{key}"] = value
    rows.append(row)

combined_df = pd.DataFrame(rows)
combined_csv = CONTEXT_DIR / "combined_summary.csv"
combined_df.to_csv(combined_csv, index=False, encoding="utf-8-sig")
print("saved", combined_csv)
display(combined_df)
sync_to_drive(combined_csv)


## figure 생성

`combined_summary.csv`와 judge output을 바탕으로 결과 그림을 `figures/` 아래에 저장합니다.


In [ ]:
plot_from_input_dir(
    input_dir=CONTEXT_DIR,
    output_dir=CONTEXT_DIR / "figures",
)
print("saved figures ->", CONTEXT_DIR / "figures")
sync_to_drive(CONTEXT_DIR / "figures")
